In [ ]:
# For the moment, no significantly interesting results
from clustering import Clustering
from data_analysis import DataAnalyse
from format_data import DataFormatter
import matplotlib
from matplotlib import pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
# initialis
N_CLUSTERS = 2
LEAF = "foliolo7_enves_a10"
cluster = Clustering(n_clusters=N_CLUSTERS, leaf=LEAF)
data_analyst = DataAnalyse()

In [ ]:
# clustering on raw data
cluster.load_data()
#print(cluster.points.shape)
# normalise :
from data_processing import ImageCleaner
cluster.points = ImageCleaner().normalise_image(LEAF)
_, mask = DataFormatter().leaf_mask_data(LEAF, return_mask=True)
cluster.points = cluster.points[mask]

print(cluster.points.shape)
cluster.transform_points()
print(cluster.points.shape)
cluster.cluster(cluster.points)

In [ ]:
plt.scatter(cluster.points[:, 0], cluster.points[:, 1], s=0.1)
plt.title("Point distribution for 2 features on LEAFnormalised data")

In [ ]:
cluster.plot_clusters_on_leaf()

In [ ]:
# raw data to 2D
# Choose TNSE or UMAP
# cluster.load_tnse()
# Create UMAP instance with default parameters
cluster.load_umap()

In [ ]:
# with current computed label (fit on raw data)
cluster.plot_embedded()

In [ ]:
# Using actual labels
cluster.plot_embedded(labels=cluster.real_labels)

In [ ]:
# plot with distance
alt_dist = data_analyst.alternative_distance_mask(leaf=LEAF)
on_leaf_alt_dist = alt_dist[alt_dist >= -126]
cluster.plot_embedded(labels=on_leaf_alt_dist)

In [ ]:
# train clustering on tnse data, and plot tnse with the new labels
cluster.cluster(cluster.embedded_data)
cluster.plot_embedded(clustering_data="UMAP data")
# The result is not convincing.

In [ ]:
# What does it look like on the leaf
cluster.plot_clusters_on_leaf(clustering_data='UMAP data')

In [ ]:
# go nack to clustering on raw data.
cluster.cluster(cluster.points)

# Try to create subclusters out of the minority class
# Try with 3 clusters.
n2_cluster_labels = np.copy(cluster.kmeans.labels_)

# select the minority class
class0len = len(n2_cluster_labels[n2_cluster_labels == 0])
class1len = len(n2_cluster_labels[n2_cluster_labels == 1])
min_class = 0 if class0len < class1len else 1
class1_points = cluster.points[n2_cluster_labels == min_class]
# train cluster on this subdataset
cluster.n_clusters = 3
cluster.cluster(class1_points)

# new_labels : just change class 1
new_labels = np.copy(n2_cluster_labels)
# + 2 to make sure there no label the same number as the majority class
new_labels[n2_cluster_labels == min_class] = cluster.kmeans.labels_ + 2

In [ ]:
# plot
print(new_labels.shape)
print(cluster.kmeans.labels_.shape)
cluster.plot_embedded(labels=new_labels)

In [ ]:
# Then again, what does it look like on the leaf ?
cluster.plot_clusters_on_leaf(labels=new_labels)
# Not very convincing either